In [3]:
!pip install -q google-genai openpyxl

In [4]:
from google import genai
import os

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY", "YOUR_KEY_HERE"))

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Say hello"
)
print(response.text)

Hello! How can I help you today?


In [5]:
import json
from pathlib import Path

EXTRACT_PROMPT = """
Extract the following fields from this invoice and return ONLY valid JSON, no explanation:
{
  "supplier_name": "",
  "gstin": "",
  "invoice_number": "",
  "invoice_date": "",
  "invoice_currency": "",
  "base_amount": 0.0,
  "tax_amount": 0.0,
  "total_amount": 0.0,
  "tax_rate_percent": 0.0
}
Rules:
- invoice_date must always be in DD-MM-YYYY format regardless of how it appears in the invoice
- If a field is missing or unclear, use null
- All amounts must be numbers, not strings
- tax_amount is the SUM of all tax components (CGST + SGST + IGST combined)
- tax_rate_percent is the total effective GST rate (e.g. 9% CGST + 9% SGST = 18% total)
"""

def extract_invoice(pdf_path: str) -> dict:
    pdf_bytes = Path(pdf_path).read_bytes()

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=[
            EXTRACT_PROMPT,
            {"inline_data": {"mime_type": "application/pdf", "data": pdf_bytes}}
        ]
    )

    # Clean response and parse JSON
    text = response.text.strip().strip("```json").strip("```").strip()
    return json.loads(text)

In [6]:
import re

def validate_invoice(data: dict, seen_invoice_numbers: set) -> list:
    issues = []

    # Check 1: GSTIN format
    gstin = data.get("gstin") or ""
    gstin_pattern = r"^\d{2}[A-Z]{5}\d{4}[A-Z]{1}[A-Z\d]{1}Z[A-Z\d]{1}$"
    oidar_pattern = r"^99[A-Z0-9]{13}$"   # foreign OIDAR suppliers
    if not re.match(gstin_pattern, gstin) and not re.match(oidar_pattern, gstin):
        issues.append(f"Invalid GSTIN: '{gstin}'")

    # Check 2: Math — base + tax should equal total
    base  = data.get("base_amount") or 0
    tax   = data.get("tax_amount") or 0
    total = data.get("total_amount") or 0
    if abs((base + tax) - total) > 0.5:
        issues.append(f"Amount mismatch: {base} + {tax} = {base+tax}, but total is {total}")

    # Check 3: Duplicate invoice number
    inv_no = data.get("invoice_number")
    if inv_no in seen_invoice_numbers:
        issues.append(f"Duplicate invoice number: {inv_no}")
    else:
        seen_invoice_numbers.add(inv_no)

    # Check 4: Missing critical fields
    for field in ["supplier_name", "gstin", "invoice_number", "invoice_date"]:
        if not data.get(field):
            issues.append(f"Missing field: {field}")

    return issues

In [7]:
import glob
import openpyxl

def run_pipeline(invoice_folder: str, output_file: str):
    pdf_files = glob.glob(f"{invoice_folder}/*.pdf")
    seen_numbers = set()
    results = []

    for pdf_path in pdf_files:
        filename = pdf_path.split("/")[-1]
        print(f"Processing: {filename}")

        try:
            extracted = extract_invoice(pdf_path)
        except Exception as e:
            results.append({"file": filename, "supplier_name": None,
                            "gstin": None, "invoice_number": None,
                            "invoice_date": None, "base_amount": None,
                            "tax_amount": None, "total_amount": None,
                            "tax_rate_percent": None, "issues": f"Extraction failed: {e}"})
            continue

        issues = validate_invoice(extracted, seen_numbers)
        row = {
            "file": filename,
            **extracted,
            "issues": "; ".join(issues) if issues else "✓ Clean"
        }
        results.append(row)
        print(f"  → {row['issues']}")

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Invoices"
    if results:
        headers = list(results[0].keys())
        ws.append(headers)
        for row in results:
            ws.append([row.get(h) for h in headers])

    wb.save(output_file)
    print(f"\n✓ Done. Saved to {output_file}")

# Run it
run_pipeline("/content", "/content/invoices_output.xlsx")

Processing: Indigo_Invoice_2.pdf
  → Invalid GSTIN: '09AABCIS2726B1Z0'
Processing: Indigo Invoice.pdf
  → ✓ Clean
Processing: Icici Invoices.pdf
  → ✓ Clean
Processing: Bill-9749053.pdf
  → ✓ Clean
Processing: Booking Invoice.pdf
  → Missing field: supplier_name
Processing: A_Invoice.pdf
  → ✓ Clean
Processing: My_Gate_Maintenace.pdf
  → ✓ Clean
Processing: My Gate Invoice.pdf
  → Duplicate invoice number: VARI-26-27-3241
Processing: Audit_Fees_2026.pdf
  → ✓ Clean
Processing: Mr. Ravi Sehgal.pdf
  → Amount mismatch: 80071.64 + 14150.88 = 94222.52, but total is 94224.52; Missing field: supplier_name

✓ Done. Saved to /content/invoices_output.xlsx
